# DIP reproduction from scratch — driver notebook

You write the code in `repro/` (in the repo, branch `claude/epic-ride-rnqbk4`); this notebook only loads it and runs the checks.

| Milestone | You write | Check |
|---|---|---|
| **M1** | `repro/data.py` | image sizes, cropping, LR size, bicubic baseline on baby = 30.73 dB |
| **M2** | `repro/metrics.py` | PSNR matches scikit-image and `evaluate_set5.py`; the paper's Bicubic row |
| M3 | `repro/train.py` (DIP loop) | short run: loss goes down, output 512×512 in [0, 1] |
| M4 | full run on baby | target 31.49 dB (Y channel, 4-px border) |
| M5 | `repro/wandb_log.py` | curves + image panels every 100 iterations in W&B |
| M6 | all 5 images × 3 seeds | DIP row: 31.49 / 31.80 / 26.23 / 31.04 / 28.93, mean 29.89 |

**Editing workflow:** edit the files in `repro/` (on GitHub, locally, or in Colab's file browser under `/content/deep-image-prior/repro/`), push, then re-run the Setup cell. If you edit directly in Colab, just re-run the check cells: they reload your code each time.

## Setup (CPU is enough for M1 and M2)

In [ ]:
import os, sys, subprocess, importlib
import numpy as np
from pathlib import Path
REPO_URL, BRANCH = "https://github.com/lakshjain7/deep-image-prior.git", "claude/epic-ride-rnqbk4"
REPO_DIR = Path("/content/deep-image-prior")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "origin", BRANCH], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))

SET5_DIR = Path("/content/Set5")
if not any(SET5_DIR.rglob("baby.png")):
    import urllib.request, tarfile
    SET5_DIR.mkdir(parents=True, exist_ok=True)
    tmp = SET5_DIR / "Set5_HR.tar.gz"
    urllib.request.urlretrieve("https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz", tmp)
    tarfile.open(tmp).extractall(SET5_DIR)
print("Set5 files:", sorted(p.name for p in SET5_DIR.rglob("*.png")))

## M1 check: `repro/data.py`

In [ ]:
import repro.data, repro.metrics, repro.checks
for m in (repro.data, repro.metrics, repro.checks): importlib.reload(m)
repro.checks.check_m1(SET5_DIR)

## M2 check: `repro/metrics.py` and the paper's Bicubic row

In [ ]:
for m in (repro.data, repro.metrics, repro.checks): importlib.reload(m)
repro.checks.check_m2(SET5_DIR)

## Look at the images (optional): HR, LR and bicubic for one image

In [ ]:
import matplotlib.pyplot as plt
from repro import data
NAME = "baby"
hr = data.crop_to_multiple(data.load_hr(data.find_set5(SET5_DIR)[NAME]), 32)
lr = data.downsample(hr, 4, "pil-lanczos")
up = data.upsample_bicubic(lr, hr.shape[:2])
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
for a, (t, im) in zip(ax, [("HR (truth)", hr), (f"LR input {lr.shape[1]}x{lr.shape[0]}", lr), ("Bicubic upsampling", up)]):
    a.imshow(np.clip(im, 0, 1), interpolation="nearest"); a.set_title(t); a.axis("off")
plt.show()